# Tactical Speech Recognition - EDA & Exploration

이 노트북은 전장 환경 음성 인식 모델 개발을 위한 **데이터 탐색(EDA), 오디오 시각화/재생, 열화 파이프라인 분석 및 추론 결과 비교**를 위한 탐색용 노트북입니다.

모듈화된 `src/` 패키지의 함수들을 활용하여 실험 및 인터랙티브 분석을 수행합니다.

In [ ]:
# 1. 환경 설정 및 모듈 경로 추가
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

# 프로젝트 루트 경로를 sys.path에 추가
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data_loader import load_speech_dataset
from src.preprocess import TacticalAudioDegrader
from src.model import load_whisper_model_and_processor
import torch

## 1. Clean 데이터셋 로드 및 원본 오디오 확인
LibriSpeech clean 데이터셋의 샘플 오디오를 청취하고 파형을 시각화합니다.

In [ ]:
# Clean 데이터셋 로드
dataset = load_speech_dataset("openslr/librispeech_asr", "clean", split="validation[:10]")
sample = dataset[0]
audio = sample["audio"]
sr = audio["sampling_rate"]
raw_audio = audio["array"]

print(f"[텍스트 정답]: {sample['text']}")
print(f"샘플링 레이트: {sr} Hz, 길이: {len(raw_audio) / sr:.2f}초")

# 원본 오디오 재생
display(Audio(raw_audio, rate=sr))

In [ ]:
# 원본 오디오 파형 시각화
time_axis = np.linspace(0, len(raw_audio) / sr, len(raw_audio))
plt.figure(figsize=(10, 3))
plt.plot(time_axis, raw_audio, color='royalblue', lw=0.8)
plt.title("Clean Speech Waveform")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid(True, alpha=0.3)
plt.show()

## 2. 전장 환경 열화(노이즈 + 대역폭 제한) 시뮬레이션 및 비교
구형 무전기 대역폭(300Hz ~ 2500Hz) 필터와 가우시안 노이즈(-5dB ~ 0dB SNR)가 결합된 열화 오디오를 청취하고 파형/스펙트로그램을 비교합니다.

In [ ]:
# 열화 파이프라인 생성 및 단일 샘플 적용
degrader = TacticalAudioDegrader(min_snr_db=-5.0, max_snr_db=0.0, central_freq=1400.0, q=0.707)
noisy_audio = degrader.degrade_audio(raw_audio, sample_rate=sr)

# 열화된 오디오 재생
print("전장 환경 열화 오디오 재생:")
display(Audio(noisy_audio, rate=sr))

In [ ]:
# Clean vs Degraded 오디오 파형 및 스펙트로그램 비교 시각화
fig, axs = plt.subplots(2, 2, figsize=(14, 6))

# Waveforms
axs[0, 0].plot(time_axis, raw_audio, color='royalblue', lw=0.7)
axs[0, 0].set_title("Original Clean Waveform")
axs[0, 0].grid(True, alpha=0.3)

axs[0, 1].plot(time_axis, noisy_audio, color='crimson', lw=0.7)
axs[0, 1].set_title("Tactical Degraded Waveform")
axs[0, 1].grid(True, alpha=0.3)

# Spectrograms
axs[1, 0].specgram(raw_audio, Fs=sr, NFFT=512, noverlap=256, cmap='viridis')
axs[1, 0].set_title("Clean Spectrogram")
axs[1, 0].set_ylabel("Frequency (Hz)")

axs[1, 1].specgram(noisy_audio, Fs=sr, NFFT=512, noverlap=256, cmap='viridis')
axs[1, 1].set_title("Tactical Degraded Spectrogram (Bandpass + Noise)")
axs[1, 1].set_ylabel("Frequency (Hz)")

plt.tight_layout()
plt.show()

## 3. Whisper 모델 로드 및 Zero-shot 추론 결과 탐색
전장 환경 데이터에서 사전학습된 Whisper 모델의 음성 인식 성능을 확인합니다.

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
model, processor = load_whisper_model_and_processor("openai/whisper-small", device=device)

# 단일 열화 샘플에 대한 추론 수행
inputs = processor(noisy_audio, sampling_rate=sr, return_tensors="pt").input_features.to(device)
with torch.no_grad():
    predicted_ids = model.generate(inputs)

pred_text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip().upper()
gt_text = sample["text"].strip().upper()

print("=" * 60)
print(f"[정답 Ground Truth]:\n{gt_text}")
print("-" * 60)
print(f"[Zero-shot 예측]:\n{pred_text}")
print("=" * 60)

## 4. LoRA 어댑터 설정 확인
파라미터 효율적 미세조정(PEFT)을 위해 LoRA를 적용하고 학습 대상 파라미터 비율을 확인합니다.

In [ ]:
from src.model import setup_lora_model

peft_model = setup_lora_model(model, r=8, lora_alpha=32, target_modules=["q_proj", "v_proj"])
# 학습 가능한 파라미터 수 및 비중 출력 확인

## 5. 전체 파이프라인 학습 및 결과 요약 안내
전체 훈련 및 평가는 터미널에서 `src/train.py`를 실행하여 수행할 수 있습니다:

```bash
python src/train.py --epochs 2 --batch_size 4
```

학습된 체크포인트는 `./tactical_whisper_baseline`에 저장됩니다.